**Amrit Dhillon**
**GSB545 Homework #3 GradientBoosting Model Notebook**

**Model 1 - Baseline LightGBM**

In [2]:
from lightgbm import LGBMClassifier
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cat_cols = X.select_dtypes(include='object').columns.tolist()


In [4]:
X_lgb = X.copy()
X_test_lgb = X_test.copy()

for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

lgb_base = LGBMClassifier(
    objective='multiclass',
    random_state=42
)

lgb_cv_scores = cross_val_score(
    lgb_base,
    X_lgb,
    y_encoded,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=1
)

print("LightGBM baseline CV balanced_accuracy:", lgb_cv_scores.mean())
print("LightGBM CV std:", lgb_cv_scores.std())

# fit full train and make submission
lgb_base.fit(X_lgb, y_encoded)
lgb_preds_encoded = lgb_base.predict(X_test_lgb)
lgb_preds = label_encoder.inverse_transform(lgb_preds_encoded)

submission_lgb = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': lgb_preds
})

submission_lgb.to_csv(
    '/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_lgb_baseline.csv',
    index=False
)

submission_lgb.head()

print(f"LightGBM baseline CV balanced_accuracy mean: {lgb_cv_scores.mean():.6f}")
print(f"LightGBM baseline CV balanced_accuracy std:  {lgb_cv_scores.std():.6f}")
print("Fold scores:", [round(s, 6) for s in lgb_cv_scores])



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007266 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2704
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005277 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2705
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Star

**Model 2 - Baseline HistGradientBoost**

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hgb_base = HistGradientBoostingClassifier(
    random_state=42
)

hgb_cv_scores = cross_val_score(
    hgb_base,
    X,
    y_encoded,
    cv=skf,
    scoring='balanced_accuracy'
)

print(f"HistGradientBoosting baseline CV balanced_accuracy mean: {hgb_cv_scores.mean():.6f}")
print(f"HistGradientBoosting baseline CV balanced_accuracy std:  {hgb_cv_scores.std():.6f}")
print("Fold scores:", [f"{s:.6f}" for s in hgb_cv_scores])

hgb_base.fit(X, y_encoded)

hgb_preds_encoded = hgb_base.predict(X_test).astype(int)
hgb_preds = label_encoder.inverse_transform(hgb_preds_encoded)

submission_hgb = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': hgb_preds
})

submission_hgb.to_csv(
    '/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_hgb_baseline.csv',
    index=False
)

submission_hgb.head()

HistGradientBoosting baseline CV balanced_accuracy mean: 0.961034
HistGradientBoosting baseline CV balanced_accuracy std:  0.000911
Fold scores: ['0.959364', '0.962095', '0.961438', '0.960978', '0.961293']


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


**Tuning on Different Sets of Hyperparameters**

In [6]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')


for df in [train, test]:
    #engineered features I used in XGBoost model last week
    df['Moisture_to_Heat_Ratio'] = df['Soil_Moisture'] / (df['Temperature_C'] + 1)
    df['Water_Availability_Index'] = (df['Rainfall_mm'] + df['Previous_Irrigation_mm']) / (df['Field_Area_hectare'] + 1)
    df['Climate_Stress_Index'] = (df['Temperature_C'] * df['Sunlight_Hours'] * (1 + df['Wind_Speed_kmh'] / 25)) / (df['Humidity'] + 1)

    #new features I want to see impact of
    df['Temp_x_Humidity'] = df['Temperature_C'] * df['Humidity']
    df['Rainfall_x_SoilMoisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
    df['Sunlight_x_Temp'] = df['Sunlight_Hours'] * df['Temperature_C']


X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [7]:
X_lgb = X.copy()
cat_cols = X_lgb.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype('category')

lgb_params = [
    {"name": "LGB_Set_1", "params": {
        "objective": "multiclass", "random_state": 42, "verbose": -1
    }},
    {"name": "LGB_Set_2", "params": {
        "objective": "multiclass", "random_state": 42, "verbose": -1,
        "learning_rate": 0.03, "n_estimators": 500, "num_leaves": 63,
        "subsample": 0.8, "colsample_bytree": 0.8
    }},
    {"name": "LGB_Set_3", "params": {
        "objective": "multiclass", "random_state": 42, "verbose": -1,
        "learning_rate": 0.15, "n_estimators": 180, "num_leaves": 15, "max_depth": 6,
        "min_child_samples": 50, "reg_alpha": 0.5, "reg_lambda": 1.0
    }},
]

lgb_results = []
for cfg in lgb_params:
    model = LGBMClassifier(**cfg["params"])
    scores = cross_val_score(model, X_lgb, y_encoded, cv=skf, scoring='balanced_accuracy', n_jobs=1)
    lgb_results.append({
        "model": cfg["name"],
        "cv_mean_bal_acc": scores.mean(),
        "cv_std": scores.std(),
        "fold_scores": [round(s, 6) for s in scores]
    })

lgb_results_df = pd.DataFrame(lgb_results).sort_values("cv_mean_bal_acc", ascending=False).reset_index(drop=True)
print("LightGBM results")
display(lgb_results_df)

LightGBM results


,model,cv_mean_bal_acc,cv_std,fold_scores
0,LGB_Set_2,0.962233,0.000685,"[0.961504, 0.962928, 0.962955, 0.961354, 0.962..."
1,LGB_Set_3,0.961666,0.000758,"[0.96103, 0.96264, 0.962378, 0.96066, 0.961622]"
2,LGB_Set_1,0.960603,0.001000,"[0.959502, 0.961832, 0.961759, 0.959724, 0.960..."


In [8]:
X_hgb = pd.get_dummies(X)
X_test_hgb = pd.get_dummies(X_test)
X_hgb, X_test_hgb = X_hgb.align(X_test_hgb, join='left', axis=1, fill_value=0)

hgb_params = [
    {"name": "HGB_Set_1", "params": {
        "random_state": 42
    }},
    {"name": "HGB_Set_2", "params": {
        "random_state": 42, "learning_rate": 0.03, "max_iter": 300,
        "max_depth": 10, "min_samples_leaf": 10
    }},
    {"name": "HGB_Set_3", "params": {
        "random_state": 42, "learning_rate": 0.2, "max_iter": 150,
        "max_depth": 6, "min_samples_leaf": 50, "l2_regularization": 1.0
    }},
]

hgb_results = []
for cfg in hgb_params:
    model = HistGradientBoostingClassifier(**cfg["params"])
    scores = cross_val_score(model, X_hgb, y_encoded, cv=skf, scoring='balanced_accuracy', n_jobs=1)
    hgb_results.append({
        "model": cfg["name"],
        "cv_mean_bal_acc": scores.mean(),
        "cv_std": scores.std(),
        "fold_scores": [round(s, 6) for s in scores]
    })

hgb_results_df = pd.DataFrame(hgb_results).sort_values("cv_mean_bal_acc", ascending=False).reset_index(drop=True)
print("HistGradientBoosting results")
display(hgb_results_df)

HistGradientBoosting results


,model,cv_mean_bal_acc,cv_std,fold_scores
0,HGB_Set_2,0.961408,0.000804,"[0.960023, 0.96254, 0.961414, 0.961583, 0.961477]"
1,HGB_Set_3,0.960986,0.000889,"[0.959785, 0.962401, 0.961125, 0.961272, 0.960..."
2,HGB_Set_1,0.960514,0.001333,"[0.958627, 0.962736, 0.960187, 0.96013, 0.960889]"


In [9]:
#simple script I generated to get the submission .csv for each of the best two models that I tuned for the boosting methods

train_sub = train.copy()
test_sub = test.copy()

for df in [train_sub, test_sub]:
    df['Moisture_to_Heat_Ratio'] = df['Soil_Moisture'] / (df['Temperature_C'] + 1)
    df['Water_Availability_Index'] = (df['Rainfall_mm'] + df['Previous_Irrigation_mm']) / (df['Field_Area_hectare'] + 1)
    df['Climate_Stress_Index'] = (df['Temperature_C'] * df['Sunlight_Hours'] * (1 + df['Wind_Speed_kmh'] / 25)) / (df['Humidity'] + 1)
    df['Temp_x_Humidity'] = df['Temperature_C'] * df['Humidity']
    df['Rainfall_x_SoilMoisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
    df['Sunlight_x_Temp'] = df['Sunlight_Hours'] * df['Temperature_C']

X_full = train_sub.drop(columns=['Irrigation_Need', 'id'])
y_full = train_sub['Irrigation_Need']
X_test_full = test_sub.drop(columns=['id'])

le = LabelEncoder()
y_full_enc = le.fit_transform(y_full)

best_lgb_name = lgb_results_df.loc[0, "model"]
best_hgb_name = hgb_results_df.loc[0, "model"]

best_lgb_params = next(cfg["params"] for cfg in lgb_params if cfg["name"] == best_lgb_name)
best_hgb_params = next(cfg["params"] for cfg in hgb_params if cfg["name"] == best_hgb_name)

X_lgb = X_full.copy()
X_test_lgb = X_test_full.copy()

cat_cols = X_lgb.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    all_cats = pd.Categorical(pd.concat([X_lgb[col], X_test_lgb[col]], axis=0)).categories
    X_lgb[col] = pd.Categorical(X_lgb[col], categories=all_cats)
    X_test_lgb[col] = pd.Categorical(X_test_lgb[col], categories=all_cats)

best_lgb_model = LGBMClassifier(**best_lgb_params)
best_lgb_model.fit(X_lgb, y_full_enc)

lgb_pred_enc = best_lgb_model.predict(X_test_lgb).astype(int)
lgb_pred = le.inverse_transform(lgb_pred_enc)

submission_lgb_best = pd.DataFrame({
    "id": test_sub["id"],
    "Irrigation_Need": lgb_pred
})

submission_lgb_best.to_csv(
    "/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_lgb_best.csv",
    index=False
)

X_hgb = pd.get_dummies(X_full)
X_test_hgb = pd.get_dummies(X_test_full)
X_hgb, X_test_hgb = X_hgb.align(X_test_hgb, join='left', axis=1, fill_value=0)

best_hgb_model = HistGradientBoostingClassifier(**best_hgb_params)
best_hgb_model.fit(X_hgb, y_full_enc)

hgb_pred_enc = best_hgb_model.predict(X_test_hgb).astype(int)
hgb_pred = le.inverse_transform(hgb_pred_enc)

submission_hgb_best = pd.DataFrame({
    "id": test_sub["id"],
    "Irrigation_Need": hgb_pred
})

submission_hgb_best.to_csv(
    "/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_hgb_best.csv",
    index=False
)

print("Saved:")
print("- submission_lgb_best.csv")
print("- submission_hgb_best.csv")

submission_lgb_best.head(), submission_hgb_best.head()


Saved:
- submission_lgb_best.csv
- submission_hgb_best.csv


(       id Irrigation_Need
 0  630000             Low
 1  630001             Low
 2  630002             Low
 3  630003             Low
 4  630004             Low,
        id Irrigation_Need
 0  630000             Low
 1  630001             Low
 2  630002             Low
 3  630003             Low
 4  630004             Low)